# 파트 2. 구조화 출력과 시스템 연동

파트 1에서는 모델의 응답을 JSON 한 줄로 받아 `json.loads`로 읽었습니다. 모델이 지시를 따라 주었기 때문에 읽혔을 뿐이고, 형식이 지켜진다는 보장은 없었습니다. 이 파트에서는 문의 100건의 분류 결과를 다른 사람이나 다른 시스템이 쓸 수 있는 데이터로 만듭니다. 응답의 형식을 API 수준에서 보장받고, 받은 값을 검증하고, 엑셀 파일과 DB에 저장합니다. 마지막에는 "이 결과를 다른 사람에게 넘겨도 되는가"라는 질문에 숫자로 답합니다.

이 파트를 마치면 다음을 할 수 있습니다.

1. 프롬프트로 JSON을 요청하는 방식과 구조화 출력의 차이를 설명하고, 구조화 출력이 보장하는 것과 보장하지 않는 것을 구분합니다.
2. 결과 테이블의 열에서 출발해 스키마를 설계합니다. 자료형, 허용값, 값이 없을 때의 처리를 정합니다.
3. 한 건을 처리하는 함수를 배치 처리로 확장하고, 실패한 건의 사유를 결과에 남깁니다.
4. 스키마 검증, 업무 규칙 검증, 재실행 비교, 정답 라벨 비교가 각각 무엇을 확인하는지 구분하고 코드로 구현합니다.
5. 결과를 자동 처리, 사람 확인, 실패로 나누고, 자동 처리한 결과 가운데 틀린 건이 얼마나 되는지 숫자로 말합니다.
6. 결과를 엑셀과 DB에 저장하되, 같은 배치를 다시 실행해도 행이 중복되지 않게 합니다.

예제의 실행 결과는 Claude 모델로 실행한 값이고, 학습 노트북을 실행한 출력을 그대로 실었습니다. 본문에서 "이번 실행"은 교재에 실린 출력을 가리키고, 2026-11-19로 날짜를 밝힌 실행은 그 전에 실행한 기록입니다. 모델의 응답과 건수는 실행할 때마다 조금씩 다릅니다. 예제의 재료는 파트 1에서 쓴 문의 100건(`문의_예제.csv`)과 정답 파일입니다.

## 이 파트에서 만드는 것

> **요점**
>
> 문의 목록의 행마다 모델이 분류한 값을 열로 붙여 결과 테이블을 만듭니다. 문의 한 건이 결과 테이블의 한 행입니다.

입력은 파트 1에서 쓴 문의 목록(`문의_예제.csv`)입니다. 문의 한 건이 한 행이고, 열은 `id`와 `문의내용` 두 개입니다.

| id | 문의내용 |
| --- | --- |
| Q001 | 3호기 PM 주기 문서는 어디에서 볼 수 있나요 |
| Q002 | 1호기 로드락 도어 인터록 알람이 세 번 연속 발생했습니다 |
| Q033 | E-201 알람이 났을 때 임시 조치 방법을 알려 주세요 |

출력은 결과 테이블입니다. 문의 목록의 행마다 열을 더 붙인 것입니다. 모델이 분류한 값(`category`, `urgent`)과, 그 값을 사람이 확인해야 하는지 나타내는 상태(`status`)가 붙습니다.

| id | 문의내용 | category | urgent | status |
| --- | --- | --- | --- | --- |
| Q001 | 3호기 PM 주기 문서는 … | 문서안내 | False | auto |
| Q002 | 1호기 로드락 도어 … | 긴급 | True | auto |
| Q033 | E-201 알람이 났을 때 … | 긴급 | True | review |

`status`의 `auto`는 사람이 보지 않고 자동 처리하는 건, `review`는 사람이 확인하는 건을 뜻합니다. 위 값은 2026-11-19에 이 파트의 예제를 실행해 얻은 결과의 일부이므로, 교재에 실린 예제 2-9의 출력과 다를 수 있습니다. 어느 건이 `review`가 되는지는 실행마다 달라질 수 있습니다(예제 2-8). 결과 테이블은 마지막에 엑셀 파일과 DB에 저장합니다. 문의 목록에서 결과 테이블까지 가는 순서는 다음과 같고, 이 파트의 절도 이 순서를 따릅니다.

1. 모델이 문의 한 건마다 `category`와 `urgent`를 정해진 구조로 돌려주게 합니다. 여기에 JSON 스키마와 구조화 출력을 씁니다.
2. 문의 100건을 배치로 처리합니다. 실패한 건은 사유와 함께 남깁니다.
3. 받은 값을 검증하고, 사람이 확인할 건을 골라 `status`를 붙입니다.
4. 결과 테이블을 엑셀 파일과 DB에 저장합니다.

이 파트에서 "문의 목록"은 입력 파일을, "결과 테이블"은 위의 출력을 가리킵니다. DB 안에 만드는 테이블은 "DB 테이블"이라고 구분해 씁니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: 파트 3의 예제는 python -m pip install sentence-transformers scikit-learn도 필요합니다
DATA = 'data/2일차'
SYSTEM = open(f'{DATA}/분류_규칙.txt', encoding='utf-8').read()

# 100만 토큰당 달러 (입력, 출력). Anthropic 공개 가격, 2026-11-03 확인.
# 소형은 입력이 10만 토큰 이하인 요청의 단가입니다. 넘으면 (0.50, 2.50)입니다.
# 사내 API를 쓴다면 사내 과금 기준으로 바꿉니다. 과금이 없으면 (0, 0)으로 둡니다.
PRICE = {'소형': (0.10, 0.50), '기본': (2.00, 10.00), '대형': (4.00, 20.00)}


def cost_usd(input_tokens, output_tokens, model='기본'):
    """토큰 수와 단가로 요금(달러)을 계산합니다."""
    input_price, output_price = PRICE[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 응답을 구조화된 데이터로 받아야 하는 이유

> **요점**
>
> 응답에서 분류값을 꺼내 조건문에 쓰거나 결과 테이블에 저장하려면, 응답을 정해진 구조로 받아야 합니다. 프롬프트로 형식을 요청하면 대부분 지켜지지만, 지켜지지 않는 경우를 막지는 못합니다.

결과 테이블의 한 행을 채우려면 문의 한 건마다 `category`와 `urgent` 값이 필요합니다. 그런데 모델이 돌려주는 응답은 문자열입니다. 그래서 코드가 이 문자열을 파싱해 파이썬 값으로 변환한 뒤 결과 테이블에 넣습니다. 아래 그림은 문의 한 건이 결과 테이블의 한 행이 되기까지의 흐름입니다.



*모델의 응답은 문자열입니다. 2번에서 3번으로 가는 주황색 화살표가 파싱이고, 응답이 정해진 형식에서 한 글자만 벗어나도 파싱이 실패합니다.*

JSON은 이름과 값의 쌍을 중괄호로 묶어 적는 텍스트 형식입니다. 이름과 값 한 쌍을 필드라고 하고, 중괄호로 묶은 전체를 객체라고 합니다. 파싱은 텍스트를 해석해 프로그램이 다루는 자료 구조로 바꾸는 일이고, JSON 텍스트는 `json.loads`가 파이썬의 dict로 바꿔 줍니다. 모델이 돌려주는 것은 언제나 문자열이므로, 그 문자열이 올바른 JSON이어야 다음 단계로 넘어갈 수 있습니다.

형식을 정하지 않으면 모델은 사람이 읽기 좋은 글로 응답합니다. 프롬프트에 "JSON 한 줄만 쓴다"고 적으면 대부분 그렇게 응답합니다.

이 파트의 예제는 `llm.ask()`로 요청을 보냅니다. 파트 1의 예제 1-10에서 만든 `ask()`와 같은 일을 하는 함수가 `llm.py`에 들어 있습니다. 질문 하나를 보내고, 재시도할 만한 오류가 나면 대기 시간을 늘리며 모두 세 번까지 보냅니다. 모델 등급(`model`)과 최대 출력 토큰(`max_tokens`)을 인자로 받고, 이 파트에서 쓸 스키마(`schema`)도 받습니다.

**예제 2-1. 형식을 정하지 않은 응답과 프롬프트로 요청한 JSON**

In [3]:
# 예제 2-1
inquiries = pd.read_csv(f'{DATA}/문의_예제.csv').fillna('')
answers = pd.read_csv(f'{DATA}/문의_예제_정답.csv').set_index('id')
LABELS = ['문서안내', '절차안내', '데이터요청', '긴급', '기타']


def prompt_of(row):
    """문의 한 건을 모델에 보낼 글로 만듭니다."""
    return f"id: {row['id']}\n문의: {row['문의내용']}"


row = inquiries.iloc[1]
print(prompt_of(row))
print('----')
reply = llm.ask('다음 문의를 분류해 주세요.\n' + row['문의내용'], model='소형')   # 형식을 정하지 않았습니다
print(reply.text)

id: Q002
문의: 1호기 로드락 도어 인터록 알람이 세 번 연속 발생했습니다
----


## 문의 분류

| 항목 | 분류 |
|---|---|
| **대분류** | 설비 장애 (알람/인터록 이상) |
| **중분류** | 로드락(Load Lock) 도어 인터록 |
| **대상 설비** | 1호기 |
| **증상** | 인터록 알람 반복 발생 (3회 연속) |
| **긴급도** | 높음 (반복 알람은 설비 정지 또는 공정 영향 가능성이 있음) |
| **우선순위** | 1순위 (즉시 확인 필요) |
| **담당 부서** | 설비 엔지니어링 / 설비 유지보수팀 |
| **문의 유형** | 장애 신고 (원인 분석 및 조치 요청) |

### 참고 사항
- **반복성 알람**이므로 단순 일시 오류보다 센서, 도어 구동부, 밸브, 인터록 신호 회로 등 **하드웨어 또는 신호 이상**을 의심할 수 있습니다.
- 재발 이력이 있다면 **반복 장애(Repeat Issue)**로 별도 태깅하는 것을 권장합니다.
- 공정 중 발생했다면 **웨이퍼/기판 영향 여부** 확인이 함께 필요할 수 있습니다.


같은 문의를 파트 1의 분류 규칙(`SYSTEM`)과 함께 보냅니다. 규칙의 끝에 "JSON 한 줄만 쓴다"는 지시가 있습니다.

In [4]:
# 예제 2-1 (프롬프트로 JSON을 요청합니다)
reply = llm.ask(prompt_of(row), system=SYSTEM, model='소형')
print(repr(reply.text))
print(json.loads(reply.text))

'{"id": "Q002", "category": "긴급", "urgent": true}'
{'id': 'Q002', 'category': '긴급', 'urgent': True}


프롬프트로만 요청했을 때 올 수 있는 응답 네 가지를 파싱해 봅니다. 아래 문자열은 모델의 응답이 아니라 설명을 위해 직접 만든 것입니다.

In [5]:
# 예제 2-1 (형식이 지켜지지 않은 응답을 파싱하면 생기는 일)
samples = {
    '앞뒤에 문장이 붙음': '네, 분류해 드리겠습니다.\n{"id": "Q002", "category": "긴급", "urgent": true}',
    '코드 블록 표시가 붙음': '```json\n{"id": "Q002", "category": "긴급", "urgent": true}\n```',
    '목록에 없는 값': '{"id": "Q002", "category": "긴급/절차안내", "urgent": true}',
    '중간에 잘림': '{"id": "Q002", "category": "긴',
}
for name, text in samples.items():
    try:
        parsed = json.loads(text)
        problem = '파싱은 되지만 분류값이 목록에 없습니다' if parsed['category'] not in LABELS else '문제 없음'
    except json.JSONDecodeError as error:
        problem = f'파싱 실패({error.msg})'
    print(f'{name}: {problem}')

앞뒤에 문장이 붙음: 파싱 실패(Expecting value)
코드 블록 표시가 붙음: 파싱 실패(Expecting value)
목록에 없는 값: 파싱은 되지만 분류값이 목록에 없습니다
중간에 잘림: 파싱 실패(Unterminated string starting at)


형식을 정하지 않은 응답은 제목, 목록, 설명이 섞인 글입니다. 분류 이름도 모델이 그때그때 정하므로 코드에서 값을 꺼낼 수 없습니다.

프롬프트로 요청한 응답은 이번 실행에서 JSON 한 줄로 왔고 그대로 파싱됐습니다. 다만 이것은 모델이 지시를 따랐다는 뜻일 뿐입니다. 모델이나 문의가 바뀌면 세 번째 코드의 네 가지 경우가 섞여 나올 수 있습니다. 원본 강의안의 실행에서는 당시의 소형 모델이 응답을 코드 블록 표시로 감싸 보냈습니다.

| 경우 | 파싱 결과 | 코드에서 막는 방법 |
| --- | --- | --- |
| 앞뒤에 문장이 붙음 | 실패 | 첫 `{`부터 마지막 `}`까지만 잘라 읽습니다 |
| 코드 블록 표시가 붙음 | 실패 | 위와 같습니다 |
| 목록에 없는 값 | 성공. 오류가 나지 않아 잘못된 값이 그대로 저장됩니다 | 받은 값이 목록에 있는지 검사합니다 |
| 중간에 잘림 | 실패 | 종료 이유를 확인합니다(파트 1의 응답 검사) |

네 가지를 모두 코드로 막을 수는 있습니다. 하지만 처음부터 형식에 맞는 응답만 나오게 할 수 있다면 잘림을 뺀 세 가지(앞뒤 문장, 코드 블록 표시, 목록에 없는 값)는 생기지 않습니다. 그 기능이 구조화 출력이고, 구조화 출력에 넘기는 형식의 명세가 JSON 스키마입니다.

## JSON 스키마

> **요점**
>
> JSON 스키마는 받을 JSON의 필드 이름, 자료형, 허용값, 필수 여부를 적은 명세입니다. 스키마 자체도 JSON(파이썬에서는 dict)으로 작성합니다.

JSON 스키마는 JSON 데이터의 구조를 정의하는 표준 형식입니다. 문의 분류 결과를 받는 스키마는 다음과 같습니다.

```python
SCHEMA = {
    'type': 'object',
    'properties': {
        'id': {'type': 'string'},
        'category': {'type': 'string', 'enum': ['문서안내', '절차안내', '데이터요청', '긴급', '기타']},
        'urgent': {'type': 'boolean'},
    },
    'required': ['id', 'category', 'urgent'],
    'additionalProperties': False,
}
```

| 키워드 | 뜻 | 위 스키마에서 |
| --- | --- | --- |
| `type` | 값의 자료형. `object`, `array`, `string`, `integer`, `number`, `boolean`, `null`이 있습니다 | 결과 전체는 `object`, `urgent`는 `boolean` |
| `properties` | 객체가 가질 필드와 각 필드의 스키마 | `id`, `category`, `urgent` 세 필드 |
| `enum` | 필드가 가질 수 있는 값의 목록 | `category`는 다섯 값 중 하나 |
| `required` | 반드시 있어야 하는 필드 | 세 필드 모두 |
| `additionalProperties` | `properties`에 없는 필드를 허용할지 | `False`. 정한 필드만 받습니다 |

`boolean`은 참과 거짓 두 값만 갖는 불리언 자료형입니다. JSON에서는 `true`와 `false`로 적고, 파이썬으로 읽으면 `True`와 `False`가 됩니다.

enum은 분류처럼 정해진 값 중에서 골라야 하는 필드에 씁니다. `category`를 `string`으로만 두면 "긴급/절차안내" 같은 값도 스키마에 맞습니다. `enum`을 두면 목록의 다섯 값만 맞습니다.

구조화 출력에 쓰는 스키마는 두 가지를 지켜 작성합니다.

- 모든 객체에 `additionalProperties`를 `False`로 적습니다. 두 API 모두 이렇게 적은 스키마만 받습니다.
- 필드는 모두 `required`에 넣습니다. OpenAI 호환 API는 서버가 스키마를 강제하는 `strict` 방식("요청하는 방법" 소절)에서 이것을 요구합니다. 모든 필드를 필수로 두면 필드가 빠진 응답은 곧 잘못된 응답이므로, 값이 없어서 `null`이 온 경우와 구분됩니다.

값이 없을 수 있는 필드는 `required`에서 빼지 않고 `null`을 허용하는 방법으로 표현합니다("스키마 설계" 절).

## 구조화 출력

> **요점**
>
> 구조화 출력은 스키마에 어긋나는 토큰을 생성 단계에서 제외합니다. 그래서 정상 종료한 응답은 항상 스키마에 맞는 JSON입니다. 값이 옳은지는 보장하지 않습니다.

구조화 출력은 요청에 JSON 스키마를 넣어, 모델이 그 스키마에 맞는 JSON만 생성하게 하는 API 기능입니다.

### 형식이 보장되는 원리

모델은 토큰을 하나씩 생성하고, 매번 다음에 올 수 있는 토큰들의 확률을 계산해 그중 하나를 고릅니다(파트 1의 "응답이 매번 다른 이유"). 프롬프트로 형식을 요청하면 형식에 맞는 토큰의 확률이 높아질 뿐, 다른 토큰이 뽑힐 가능성은 남습니다.

구조화 출력을 쓰면 서버가 토큰을 고르기 전에 한 단계를 더 거칩니다. 지금까지 생성한 글에 이어 붙이면 스키마에 어긋나게 되는 토큰을 후보에서 제외하고, 남은 토큰 중에서만 고릅니다. 이 방식을 제약 디코딩이라고 합니다. 예를 들어 `"category": "`까지 생성했다면, 그다음에는 다섯 허용값 가운데 하나로 이어질 수 있는 토큰만 후보에 남습니다.

이 원리에서 구조화 출력의 성질 두 가지가 나옵니다.

- 형식은 확률이 아니라 규칙으로 지켜집니다. 앞뒤에 문장을 붙이거나 목록에 없는 값을 쓰는 일은 일어나지 않습니다.
- 서버가 제한하는 것은 형식뿐입니다. 허용값 다섯 개 가운데 어느 것을 고를지는 여전히 모델이 확률에 따라 정합니다. 틀린 분류를 고를 수 있고, 다시 실행하면 다른 값을 고를 수도 있습니다.

> **주의**
>
> Anthropic 문서는 문자열 enum 값의 대소문자를 보장하지 않는다고 밝힙니다. 띄어쓰기 뒤 낱말의 첫 글자가 대문자로 바뀌어 오는 경우가 있고, 이때도 응답은 정상 종료합니다. 영문 허용값은 대소문자를 무시하고 비교하고, 대소문자만 다른 값을 목록에 함께 두지 않습니다. 이 파트의 예제처럼 허용값이 한글이면 해당하지 않습니다. 2026-11-26 공식 문서 기준입니다.

### 요청하는 방법

두 API는 스키마를 넣는 인자의 이름과 값의 구조만 다릅니다.

| | Anthropic | OpenAI 호환 |
| --- | --- | --- |
| 스키마를 넣는 인자 | `output_config` | `response_format` |
| 응답을 읽는 곳 | `text` 블록(파트 1과 같음) | `message.content`(파트 1과 같음) |

인자에 넣는 값은 다음과 같습니다. OpenAI 호환 방식에서는 스키마에 이름을 붙이고, `strict`를 `True`로 주어야 서버가 스키마를 강제합니다.

```python
# Anthropic
output_config = {'format': {'type': 'json_schema', 'schema': SCHEMA}}

# OpenAI 호환
response_format = {'type': 'json_schema',
                   'json_schema': {'name': 'inquiry', 'schema': SCHEMA, 'strict': True}}
```

응답은 여전히 문자열입니다. 달라지는 것은 그 문자열이 스키마에 맞는 JSON이라는 점입니다.

**예제 2-2. 구조화 출력으로 받기**

In [6]:
# 예제 2-2
SCHEMA = {
    'type': 'object',
    'properties': {
        'id': {'type': 'string'},
        'category': {'type': 'string', 'enum': LABELS},
        'urgent': {'type': 'boolean'},
    },
    'required': ['id', 'category', 'urgent'],
    'additionalProperties': False,
}
# 분류 규칙에서 "출력은 … JSON 한 줄만" 이라는 형식 지시를 뺀 부분입니다. 형식은 스키마가 맡습니다.
RULES = SYSTEM.split('출력은')[0].strip()
for line in RULES.splitlines()[-2:]:              # 형식 지시가 빠졌는지 끝의 두 줄로 확인합니다
    print(line)

두 가지에 걸치면 긴급, 데이터요청, 절차안내, 문서안내 순으로 앞선 것을 고른다.
urgent 는 즉시 또는 오늘 안에 대응해 달라는 문의면 true, 아니면 false 다. 긴급은 항상 true 다.


**Anthropic**

In [7]:
# 예제 2-2 (Anthropic SDK로 직접 요청)
if llm.provider() == 'anthropic':
    import anthropic

    client = anthropic.Anthropic()
    response = client.messages.create(
        model=llm.model_name('소형'),
        max_tokens=1024,
        system=RULES,
        messages=[{'role': 'user', 'content': prompt_of(row)}],
        output_config={'format': {'type': 'json_schema', 'schema': SCHEMA}},
    )
    text = ''.join(block.text for block in response.content if block.type == 'text')
    print(repr(text))
    print('종료 이유:', response.stop_reason)
else:
    print('Anthropic을 쓰지 않으므로 건너뜁니다.')

'{"id":"Q002","category":"긴급","urgent":true}'
종료 이유: end_turn


**OpenAI 호환**

In [8]:
# 예제 2-2 (OpenAI 호환 SDK로 직접 요청)
if llm.provider() == 'openai':
    from openai import OpenAI

    client = OpenAI(base_url=os.environ['OPENAI_BASE_URL'], api_key=os.environ['OPENAI_API_KEY'])
    response = client.chat.completions.create(
        model=llm.model_name('소형'),
        max_tokens=1024,
        messages=[{'role': 'system', 'content': RULES},
                  {'role': 'user', 'content': prompt_of(row)}],
        response_format={'type': 'json_schema',
                         'json_schema': {'name': 'inquiry', 'schema': SCHEMA, 'strict': True}},
    )
    print(repr(response.choices[0].message.content))
    print('종료 이유:', response.choices[0].finish_reason)
else:
    print('OpenAI 호환 API를 쓰지 않으므로 건너뜁니다.')

OpenAI 호환 API를 쓰지 않으므로 건너뜁니다.


`llm.ask()`와 `llm.send()`에는 `schema` 인자로 넘깁니다. 지금 쓰는 API에 맞는 형식으로 바꿔 보내 줍니다.

In [9]:
# 예제 2-2 (llm.ask로 같은 요청)
reply = llm.ask(prompt_of(row), system=RULES, model='소형', schema=SCHEMA)
print(repr(reply.text))
result = json.loads(reply.text)             # 자르거나 다듬지 않고 그대로 읽습니다
print(result['category'], result['urgent'], type(result['urgent']))

'{"id":"Q002","category":"긴급","urgent":true}'
긴급 True <class 'bool'>


응답에는 코드 블록 표시도 앞뒤 문장도 없습니다. `urgent`는 문자열 `"true"`가 아니라 불리언 값(`True`)으로 왔습니다. 프롬프트에서 "JSON 한 줄만 쓴다"는 지시를 뺐는데도 형식이 지켜졌습니다. 형식은 프롬프트가 아니라 스키마가 정하기 때문입니다. 프롬프트에는 무엇을 어떤 기준으로 분류할지만 남깁니다.

필드 이름을 한글로 써도 동작합니다. 필드 이름은 뒤에서 결과 테이블과 DB 테이블의 열 이름으로 쓰이므로, 이 교재에서는 짧은 영문 이름을 씁니다.

> **보충**
>
> OpenAI 호환 서버는 구조화 출력을 지원하는 범위가 서버마다 다릅니다. `llm.py`는 환경 변수 `LLM_STRUCTURED`로 요청 방식을 고릅니다. Anthropic API를 쓴다면 설정할 필요가 없습니다.
> 
> | `LLM_STRUCTURED` | 서버가 보장하는 것 | `llm.py`가 하는 일 |
> | --- | --- | --- |
> | `json_schema`(기본값) | 스키마에 맞는 JSON | `response_format`에 스키마를 넣어 보냅니다 |
> | `json_object` | 올바른 JSON. 필드와 값은 보장하지 않습니다 | `response_format`을 `{'type': 'json_object'}`로 보내고, 스키마를 시스템 프롬프트 끝에 적어 보냅니다 |
> | `prompt` | 없음 | 스키마를 시스템 프롬프트 끝에 적어 보내기만 합니다 |
> 
> 예제 2-2의 마지막 셀이 기대와 다르게 동작하면 아래 순서로 `LLM_STRUCTURED`를 바꿉니다. 환경 변수이므로 Jupyter를 시작하기 전에 설정합니다.
> 
> - 400 오류가 나면 서버가 `json_schema` 방식을 지원하지 않는 것입니다. `json_object`로 설정하고 다시 실행합니다. 그래도 오류가 나면 `prompt`로 설정합니다.
> - 오류는 나지 않는데 응답에 코드 블록 표시나 설명 문장이 붙어 온다면, 서버가 `response_format`을 무시하고 있는 것입니다. 이때도 `prompt`로 설정합니다.
> 
> `json_object`와 `prompt` 방식에서는 서버가 스키마를 강제하지 않습니다. 받은 값을 코드에서 검증하는 일("pydantic으로 스키마를 정의하고 검증하기" 절)이 형식을 지키는 유일한 수단입니다. 사내 API에서의 동작은 이 교재를 만들 때 확인하지 못했습니다.

### 구조화 출력이 보장하지 않는 것



*청록색 상자는 구조화 출력이 지켜 주는 형식입니다. 주황색 상자의 다섯 가지는 코드에서 따로 확인해야 합니다.*

형식의 보장에는 조건이 있습니다. 응답이 정상 종료했을 때만 성립합니다.

**예제 2-3. 구조화 출력에서도 깨지는 경우**

In [10]:
# 예제 2-3 (최대 출력 토큰을 일부러 낮춥니다)
cut = llm.ask(prompt_of(row), system=RULES, model='소형', max_tokens=20, schema=SCHEMA)
print('종료 이유:', cut.stop_reason)
print(repr(cut.text))
try:
    json.loads(cut.text)
except json.JSONDecodeError as error:
    print('파싱 실패:', error.msg)

종료 이유: max_tokens
'{"id":"Q002","category":"긴급","urgent":'
파싱 실패: Expecting value


스키마에 숫자의 최솟값(`minimum`)을 넣어 봅니다.

In [11]:
# 예제 2-3 (구조화 출력이 받지 않는 키워드)
with_minimum = {
    'type': 'object',
    'properties': {'count': {'type': 'integer', 'minimum': 1}},
    'required': ['count'],
    'additionalProperties': False,
}
try:
    reply = llm.send([{'role': 'user', 'content': '알람이 세 번 울렸습니다. 횟수를 뽑아 주세요.'}],
                     model='소형', schema=with_minimum)
    print('받은 응답:', reply.text)
except llm.CallError as error:
    print(error.status, error.kind)
    start = error.detail.find('output_config')
    # 오류 문구만 출력합니다. 문구의 자리를 찾지 못하면 앞부분 200자를 출력합니다
    print(error.detail[start:].split('"')[0] if start >= 0 else error.detail[:200])

400 bad_request
output_config.format.schema: For 'integer' type, property 'minimum' is not supported


첫 번째 코드에서는 최대 출력 토큰에 도달해 JSON이 닫히기 전에 생성이 끝났습니다. 종료 이유는 `max_tokens`(OpenAI 호환은 `length`)이고 응답은 파싱되지 않습니다. 거절(`refusal`, OpenAI 호환은 `content_filter`)로 끝난 응답도 스키마와 다를 수 있습니다. OpenAI API의 구조화 출력은 모델이 거절하면 거절 문구를 `content`가 아니라 `message.refusal` 필드에 담아 보냅니다. 이 교재의 `llm.py`는 이 필드에 값이 있으면 종료 이유를 `refusal`로 돌려주므로, `parse_reply()`는 이런 응답을 "거절"로 기록합니다. 그래서 구조화 출력을 쓰더라도 파트 1의 원칙은 그대로입니다. 종료 이유가 정상 종료일 때만 응답을 씁니다.

두 번째 코드는 Anthropic API에서 400 오류로 끝났습니다. 구조화 출력은 JSON 스키마의 키워드를 모두 지원하지는 않습니다. Anthropic API는 숫자의 범위(`minimum`, `maximum`), 문자열의 길이(`minLength`, `maxLength`) 같은 제약을 요청 단계에서 거부합니다. 이런 조건은 스키마에 넣지 않고, 응답을 받은 뒤 코드에서 검사합니다. OpenAI API는 `minimum`, `maximum`을 지원하고, `minLength`, `maxLength`는 지원 목록에 없습니다. OpenAI 호환 서버는 지원하는 키워드가 서버마다 다르므로 같은 코드가 오류 없이 실행될 수도 있습니다. 두 API의 지원 범위는 2026-11-26 공식 문서 기준입니다.

> **보충**
>
> 스키마를 처음 보낼 때는 서버가 스키마를 제약 규칙으로 바꾸는 시간이 더해져 응답이 조금 늦습니다. Anthropic API는 바꾼 결과를 마지막으로 쓴 때부터 24시간 동안 보관하므로 같은 스키마의 다음 요청부터는 늦어지지 않습니다. 2026-11-26 공식 문서 기준입니다.

## 스키마 설계

> **요점**
>
> 결과 테이블에 필요한 열을 먼저 정하고, 그 열에 맞춰 스키마를 작성합니다. 골라야 하는 값은 enum으로 좁히고, 없을 수 있는 값은 null을 허용합니다.

스키마의 필드는 결과 테이블의 열이 됩니다. 그래서 프롬프트보다 스키마를 먼저 작성합니다. 결과를 받아 볼 사람이 어떤 열을 필요로 하는지 정하고, 열마다 아래 네 가지를 정합니다.

| 정할 것 | 기준 | 예 |
| --- | --- | --- |
| 자료형 | 뒤에서 그 값으로 무엇을 하는지에 맞춥니다. 조건문에 쓸 값은 `boolean`, 개수처럼 계산에 쓸 값은 `integer`로 받습니다 | `urgent`는 `boolean` |
| 허용값 | 정해진 값 중에서 고르는 필드는 `enum`으로 좁힙니다. 목록에는 어디에도 해당하지 않는 경우를 받을 값을 둡니다 | `category`의 "기타" |
| 없을 때의 값 | 원문에 없을 수 있는 값은 `null`을 허용합니다 | 장비 이름 |
| 이름 | 결과 테이블과 DB 테이블의 열 이름으로 그대로 쓸 수 있는 짧은 영문으로 짓습니다 | `id`, `category`, `urgent` |

`enum`의 목록은 답의 범위를 정합니다. 모델은 목록 안에서 반드시 하나를 골라야 하므로, 목록에 맞는 값이 없는 입력도 목록 안의 어느 한 값으로 분류됩니다. 오류도 나지 않습니다. 그래서 "기타"처럼 나머지를 받는 값이 필요하고, 그 값으로 분류된 건은 사람이 확인합니다. 실습 7에서 허용값 하나를 빼고 이 현상을 직접 확인합니다.

`null`을 허용하는 이유도 같습니다. 자료형이 `string`뿐인 필수 필드는 모델이 어떤 문자열이든 채워야 합니다. 원문에 그 값이 없으면 빈 문자열, "없음" 같은 말, 또는 원문에 없는 그럴듯한 값이 들어갑니다. `null`을 허용하면 "값이 없다"를 한 가지 방법으로 표현할 수 있고, 코드에서도 한 가지 조건으로 확인할 수 있습니다.

분류만이 아니라 문장에서 여러 값을 뽑는 일에도 같은 방법을 씁니다.

**예제 2-4. 문장에서 여러 값을 뽑기, null을 허용할 때와 허용하지 않을 때**

In [12]:
# 예제 2-4
EXTRACT = {
    'type': 'object',
    'properties': {
        'equipment': {'type': ['string', 'null']},        # 장비 이름. 문의에 없으면 null
        'symptom': {'type': 'string'},                    # 현상이나 요청을 한 줄로
        'action': {'type': 'string', 'enum': ['즉시조치', '확인요청', '자료요청', '안내']},
    },
    'required': ['equipment', 'symptom', 'action'],
    'additionalProperties': False,
}
EXTRACT_RULES = ('현장 문의에서 값을 뽑는다. equipment 는 문의에 나온 장비 이름이다. '
                 '문의에 없는 값은 null 로 둔다.')
texts = ['5호기 챔버 압력 알람이 계속 떠서 라인을 세웠습니다',
         'PM 체크리스트 최신본 위치를 알고 싶습니다',
         '넵 확인했습니다 감사합니다']
for text in texts:
    reply = llm.ask(text, system=EXTRACT_RULES, model='소형', schema=EXTRACT)
    print(json.loads(reply.text))

{'equipment': '5호기 챔버', 'symptom': '챔버 압력 알람이 계속 발생하여 라인 정지', 'action': '즉시조치'}


{'equipment': None, 'symptom': 'PM 체크리스트 최신본 위치 문의', 'action': '자료요청'}


{'equipment': None, 'symptom': '확인 응답', 'action': '안내'}


`equipment`에서 `null`을 빼고 같은 문의를 보냅니다. 규칙에서도 "없으면 null"이라는 문장을 뺍니다.

In [13]:
# 예제 2-4 (null을 허용하지 않으면)
import copy

NO_NULL = copy.deepcopy(EXTRACT)
NO_NULL['properties']['equipment'] = {'type': 'string'}
NO_NULL_RULES = '현장 문의에서 값을 뽑는다. equipment 는 문의에 나온 장비 이름이다.'
for text in texts:
    reply = llm.ask(text, system=NO_NULL_RULES, model='소형', schema=NO_NULL)
    print(repr(json.loads(reply.text)['equipment']), '←', text)

'5호기 챔버' ← 5호기 챔버 압력 알람이 계속 떠서 라인을 세웠습니다


'' ← PM 체크리스트 최신본 위치를 알고 싶습니다


'' ← 넵 확인했습니다 감사합니다


`type`에 `['string', 'null']`처럼 목록을 주면 둘 중 어느 자료형이든 허용한다는 뜻입니다. JSON의 `null`은 파이썬에서 `None`으로 읽힙니다.

`null`을 허용한 스키마에서는 장비 이름이 없는 두 문의의 `equipment`가 `None`으로 왔습니다. `null`을 허용하지 않은 스키마에서는 이번 실행에서 빈 문자열(`''`)이 왔습니다. 모델은 문자열을 반드시 채워야 했고, 가장 무난한 빈 문자열을 골랐습니다. 다른 모델이나 다른 문의에서는 "없음", "해당 없음", 또는 문의에 없는 장비 이름이 올 수 있습니다. 그러면 코드에서 "값이 없다"를 판별하는 조건을 경우마다 써야 하고, 지어낸 장비 이름은 조건으로 잡을 수도 없습니다.

첫 번째 코드의 세 번째 문의도 눈여겨봅니다. 감사 인사에는 조치가 필요 없지만 `action`이 "안내"로 왔습니다. `action`의 목록에 "해당 없음"에 해당하는 값이 없어서, 모델이 네 값 가운데 가장 가까운 것을 고른 것입니다. `enum`의 목록이 답의 범위를 정한다는 것이 이런 뜻입니다.

## pydantic으로 스키마를 정의하고 검증하기

> **요점**
>
> pydantic 클래스 하나로 스키마를 만들고, 받은 JSON을 같은 클래스로 검증합니다. 서버가 형식을 강제하지 않는 환경에서는 이 검증이 형식을 지키는 수단입니다.

스키마를 dict로 직접 작성하면 두 가지가 불편합니다. 중괄호가 깊어져 필드가 늘수록 읽기 어렵습니다. 그리고 받은 값이 스키마에 맞는지 확인하려면 검사 코드를 따로 작성해야 합니다.

pydantic은 파이썬 클래스로 데이터의 구조를 정의하는 라이브러리입니다. 클래스에 필드 이름과 자료형을 적으면 두 가지를 해 줍니다.

- 클래스에서 JSON 스키마를 만들어 줍니다(`model_json_schema()`).
- JSON 텍스트가 그 클래스의 정의에 맞는지 확인하고, 맞으면 객체로 바꿔 줍니다(`model_validate_json()`). 받은 값이 스키마에 맞는지 코드로 확인하는 이 일을 스키마 검증이라고 합니다.

| 스키마로 표현할 것 | pydantic에서 쓰는 방법 |
| --- | --- |
| 문자열, 불리언, 정수 | `str`, `bool`, `int` |
| 허용값(`enum`) | `Literal['값1', '값2']` |
| 없을 수 있는 값 | <code>str &#124; None</code> |
| 정한 필드만 허용 | `model_config = ConfigDict(extra='forbid')` |

**예제 2-5. pydantic 클래스로 스키마를 만들고 응답을 검증하기**

In [14]:
# 예제 2-5
from typing import Literal

from pydantic import BaseModel, ConfigDict, ValidationError


class Inquiry(BaseModel):
    # extra='forbid'는 정하지 않은 필드를 거부합니다(additionalProperties: false).
    # strict=True는 "yes" 같은 문자열을 True로 바꿔 받아들이지 않게 합니다.
    model_config = ConfigDict(extra='forbid', strict=True)
    id: str
    category: Literal['문서안내', '절차안내', '데이터요청', '긴급', '기타']
    urgent: bool


INQUIRY_SCHEMA = Inquiry.model_json_schema()
print(json.dumps(INQUIRY_SCHEMA, ensure_ascii=False, indent=1))

{
 "additionalProperties": false,
 "properties": {
  "id": {
   "title": "Id",
   "type": "string"
  },
  "category": {
   "enum": [
    "문서안내",
    "절차안내",
    "데이터요청",
    "긴급",
    "기타"
   ],
   "title": "Category",
   "type": "string"
  },
  "urgent": {
   "title": "Urgent",
   "type": "boolean"
  }
 },
 "required": [
  "id",
  "category",
  "urgent"
 ],
 "title": "Inquiry",
 "type": "object"
}


응답을 검증하는 함수를 만듭니다. 종료 이유를 먼저 확인하고, 그다음 스키마에 맞는지 확인합니다. 스키마에 어긋난 응답은 구조화 출력으로는 받기 어려우므로, 응답 객체를 `llm.Reply(응답 텍스트, 입력 토큰, 출력 토큰, 종료 이유, 모델 이름, 걸린 시간)`으로 직접 만들어 함수에 넣어 봅니다.

In [15]:
# 예제 2-5 (응답을 검증하는 함수)
def parse_reply(reply, model_class):
    """응답을 검증해 (객체, None) 또는 (None, 사유)를 돌려줍니다."""
    if reply.stop_reason != 'end':                     # 정상 종료가 아니면 쓰지 않습니다
        return None, {'max_tokens': '잘림', 'refusal': '거절'}.get(reply.stop_reason, '비정상 종료')
    # 서버가 형식을 강제하지 않는 환경에서는 앞뒤에 글이 붙을 수 있어 중괄호 사이만 읽습니다
    text = reply.text[reply.text.find('{'):reply.text.rfind('}') + 1]
    try:
        return model_class.model_validate_json(text), None
    except ValidationError as error:
        first = error.errors()[0]
        field = '.'.join(str(part) for part in first['loc']) or '전체'
        return None, f"스키마 불일치({field}: {first['type']})"


reply = llm.ask(prompt_of(row), system=RULES, model='소형', schema=INQUIRY_SCHEMA)
parsed, reason = parse_reply(reply, Inquiry)
print(f'{parsed}, 사유: {reason}')
print(parsed.category, parsed.urgent)

# 스키마에 어긋난 응답을 직접 만들어 넣어 봅니다
made_up = [
    '{"id": "Q002", "category": "긴급/절차안내", "urgent": true}',
    '{"id": "Q002", "category": "긴급", "urgent": "yes"}',
    '{"id": "Q002", "category": "긴급", "urgent": true, "note": "확인 바람"}',
    '네, 분류해 드리겠습니다. {"id": "Q002", "category": "긴급", "urgent": true}',
    '긴급으로 보입니다.',
]
for text in made_up:
    fake_reply = llm.Reply(text, 0, 0, 'end', 'made-up', 0.0)
    print(parse_reply(fake_reply, Inquiry)[1], '←', text[:45])

id='Q002' category='긴급' urgent=True, 사유: None
긴급 True
스키마 불일치(category: literal_error) ← {"id": "Q002", "category": "긴급/절차안내", "urgent
스키마 불일치(urgent: bool_type) ← {"id": "Q002", "category": "긴급", "urgent": "y
스키마 불일치(note: extra_forbidden) ← {"id": "Q002", "category": "긴급", "urgent": tr
None ← 네, 분류해 드리겠습니다. {"id": "Q002", "category": "긴급
스키마 불일치(전체: json_invalid) ← 긴급으로 보입니다.


pydantic이 만든 스키마에는 `title`이라는 키가 더 들어 있습니다. 설명용 키이고 형식에는 관여하지 않습니다. 나머지는 예제 2-2에서 직접 쓴 스키마와 같습니다.

`parse_reply()`는 직접 만든 다섯 개의 응답 가운데 넷에는 사유를 돌려주고, 앞에 문장이 붙은 네 번째 응답은 중괄호 사이만 읽어 통과시킵니다. 서버가 스키마를 강제하는 환경에서는 이런 응답이 오지 않으므로 검증이 항상 통과합니다. 그래도 `parse_reply()`를 거치게 두면 같은 코드가 `json_object` 방식이나 `prompt` 방식의 서버에서도 그대로 동작합니다.

> **보충**
>
> `anthropic` SDK에는 이 두 단계를 한 번에 해 주는 `client.messages.parse()`가 있습니다. `output_format`에 pydantic 클래스를 넘기면 SDK가 스키마를 만들어 보내고, 응답을 검증한 객체를 `response.parsed_output`에 넣어 줍니다. 이 교재는 두 API에서 같은 코드를 쓰기 위해 `llm.ask()`와 `parse_reply()`를 씁니다.

## 배치 처리와 실패 기록

> **요점**
>
> 한 건을 처리하는 함수가 실패의 사유까지 결과에 담아 돌려주게 만들고, 그 함수를 병렬로 실행합니다. 한 건으로 먼저 확인한 뒤 전체를 보냅니다.

파트 1의 배치 처리 흐름(입력 검사, 호출, 오류 구분, 응답 검사, 실패 기록)을 그대로 씁니다. 달라지는 것은 두 가지입니다. 호출에 스키마를 넘기고, 응답 검사를 `parse_reply()`가 맡습니다.

한 건을 처리하는 함수는 어떤 경우에도 예외를 발생시키지 않고 같은 키를 가진 dict를 돌려주게 만듭니다. 성공하면 분류 결과를, 실패하면 사유를 채웁니다. 그래야 한 건의 실패가 배치 전체를 멈추지 않고, 결과 테이블에서 실패한 건이 빠지지도 않습니다.

이때 잡는 예외의 범위를 좁게 둡니다. 모든 예외를 잡으면(`except Exception`) 내 코드의 버그로 생긴 오류도 "호출 실패"로 기록되어, 100건이 모두 같은 버그로 실패해도 알아채기 어렵습니다. 아래 코드는 API 호출의 실패(`llm.CallError`)만 잡습니다. 스키마에 맞지 않는 응답은 `parse_reply()`가 예외 없이 사유로 돌려줍니다.

**예제 2-6. 문의 100건을 구조화 출력으로 분류하기**

In [16]:
# 예제 2-6
def classify(row, model='소형'):
    """문의 한 건을 분류합니다. 실패해도 예외를 발생시키지 않고 error에 사유를 적어 돌려줍니다."""
    record = {'id': row['id'], 'answer_id': None, 'category': None, 'urgent': None,
              'error': None, 'input_tokens': 0, 'output_tokens': 0}
    try:
        reply = llm.ask(prompt_of(row), system=RULES, model=model, max_tokens=1024,
                        schema=INQUIRY_SCHEMA)
    except llm.CallError as error:
        record['error'] = f'호출 실패({error.kind})'
        return record
    record.update(input_tokens=reply.input_tokens or 0, output_tokens=reply.output_tokens or 0)
    parsed, reason = parse_reply(reply, Inquiry)
    if reason:
        record['error'] = reason
    else:
        record.update(answer_id=parsed.id, category=parsed.category, urgent=parsed.urgent)
    return record


def is_valid(text):
    """보낼 만한 문의인지 돌려줍니다. 여기서는 빈 문의만 필터링합니다."""
    return isinstance(text, str) and len(text.strip()) >= 2


to_send = [row for _, row in inquiries.iterrows() if is_valid(row['문의내용'])]
print(f'전체 {len(inquiries)}건, 보낼 건 {len(to_send)}건')
print(classify(to_send[0]))                  # 한 건으로 먼저 확인합니다

전체 100건, 보낼 건 98건


{'id': 'Q001', 'answer_id': 'Q001', 'category': '문서안내', 'urgent': False, 'error': None, 'input_tokens': 779, 'output_tokens': 57}


한 건의 결과가 기대한 대로 나오면 전체를 보냅니다.

In [17]:
# 예제 2-6 (전체를 8건씩 병렬로 보냅니다)
def run_batch(rows, model='소형', workers=8):
    """여러 건을 병렬로 분류해 결과 dict의 목록을 돌려줍니다. 순서는 rows와 같습니다."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda row: classify(row, model), rows))


def total_cost(records, model='소형'):
    return cost_usd(sum(record['input_tokens'] for record in records),
                    sum(record['output_tokens'] for record in records), model)


started = time.time()
first = run_batch(to_send)
print(f'{len(first)}건, {time.time() - started:.1f}초, 요금 ${total_cost(first):.4f}')
print('실패:', [(record['id'], record['error']) for record in first if record['error']])
print(pd.Series([record['category'] for record in first]).value_counts(dropna=False).to_dict())

98건, 17.5초, 요금 $0.0107
실패: []
{'문서안내': 25, '절차안내': 25, '긴급': 23, '데이터요청': 21, '기타': 4}


`classify()`는 모델이 돌려준 `id`를 `answer_id`에 따로 담습니다. 우리가 보낸 `id`와 같은지를 다음 절에서 검사하기 위해서입니다.

이번 실행에서는 98건을 약 18초에 처리했고 실패한 건은 없었습니다. 요금은 모두 $0.0109였습니다. 첫 번째 코드 출력의 둘째 줄을 보면 첫 건(Q001)에 입력 779토큰(분류 규칙 포함), 출력 25토큰을 썼습니다.

같은 문의가 파트 1에서는 한 건에 평균 약 490토큰이었습니다(예제 1-7). Anthropic API는 구조화 출력을 쓰면 출력 형식을 설명하는 시스템 프롬프트를 자동으로 더하므로, 입력 토큰이 그만큼 늘어납니다(공식 문서, 2026-11-26 확인). OpenAI 호환 서버의 동작은 확인하지 못했습니다.

최대 출력 토큰은 1,024토큰으로 두었습니다. 파트 1에서 300토큰으로 두었을 때 thinking이 길어진 문의 한 건이 잘렸기 때문입니다(예제 1-12). 구조화 출력에서도 thinking에 쓴 토큰은 출력 토큰에 들어갑니다.

## 검증의 네 단계

> **요점**
>
> 스키마 검증은 형식을, 업무 규칙 검증은 필드 값 사이의 모순을, 재실행 비교는 실행마다 결과가 바뀌는지를, 정답 라벨 비교는 값이 옳은지를 확인합니다. 앞의 셋을 모두 통과해도 정답이라는 뜻은 아닙니다.

"검증했다"는 말은 무엇을 확인했는지에 따라 뜻이 넷으로 나뉩니다.

| 단계 | 확인하는 것 | 필요한 것 | 확인하지 못하는 것 |
| --- | --- | --- | --- |
| 1. 스키마 검증 | 응답이 정상 종료했고 스키마에 맞는가 | 스키마 | 값이 옳은지 |
| 2. 업무 규칙 검증 | 필드 값이 서로 모순되지 않고 입력과도 맞는가 | 업무 지식으로 작성한 규칙 | 규칙으로 작성하지 못한 오류 |
| 3. 재실행 비교 | 다시 실행해도 같은 값이 나오는가 | 한 번 더 실행하는 요금 | 두 번 다 같은 값으로 틀린 건 |
| 4. 정답 라벨 비교 | 사람이 정한 정답과 같은가 | 정답 라벨이 붙은 표본 | 표본에 없는 건 |

1번과 2번은 모든 건에 요금 없이 적용할 수 있습니다. 3번은 요금이 두 배로 듭니다. 4번은 사람이 정답을 붙여야 하므로 보통 일부 표본에만 할 수 있습니다. 그래서 실제 배치에서는 1, 2, 3번으로 사람이 볼 건을 고르고, 4번으로 "자동 처리한 건 가운데 틀린 건이 얼마나 되는가"를 추정합니다.

업무 규칙은 스키마로 표현할 수 없는 조건입니다. 이 예제의 분류 규칙에는 "긴급은 항상 urgent가 true다"라는 문장이 있습니다. 스키마에서 `category`와 `urgent`는 서로 독립된 필드이므로, `category`가 "긴급"이고 `urgent`가 `false`인 응답도 스키마에는 맞습니다. 이런 조건은 코드로 검사합니다. 모델이 돌려준 `id`가 보낸 `id`와 같은지도 스키마가 확인하지 못하는 조건입니다.

**예제 2-7. 스키마 검증, 업무 규칙 검증, 정답 라벨 비교**

In [18]:
# 예제 2-7
def validate(record):
    """결과 한 건을 검사해 걸린 사유를, 통과하면 None을 돌려줍니다."""
    if record['error']:                                   # 1단계: 호출 실패, 잘림, 거절, 스키마 불일치
        return record['error']
    if record['answer_id'] != record['id']:               # 2단계: 보낸 id와 받은 id가 같은가
        return 'id 불일치'
    if record['category'] == '긴급' and not record['urgent']:
        return '긴급인데 urgent가 false'
    if record['category'] == '기타' and record['urgent']:
        return '기타인데 urgent가 true'
    return None


reasons = pd.Series([validate(record) for record in first])
print('검증 통과', reasons.isna().sum(), '/', len(first))
print('걸린 사유:', reasons.dropna().value_counts().to_dict())

검증 통과 98 / 98
걸린 사유: {}


검증을 통과한 건을 정답 라벨과 비교합니다.

In [19]:
# 예제 2-7 (검증을 통과한 건 가운데 정답과 다른 건)
def is_correct(record):
    return record['category'] == answers.loc[record['id'], 'category']


passed = [record for record in first if validate(record) is None]
wrong = [record for record in passed if not is_correct(record)]
print(f'검증 통과 {len(passed)}건, 정답과 같은 건 {len(passed) - len(wrong)}건, 다른 건 {len(wrong)}건')
text_of = dict(zip(inquiries['id'], inquiries['문의내용']))
for record in wrong:
    print(f"  {record['id']} 모델 {record['category']}, 정답 {answers.loc[record['id'], 'category']}"
          f", {answers.loc[record['id'], '난이도']}, {text_of[record['id']][:34]}")

검증 통과 98건, 정답과 같은 건 95건, 다른 건 3건
  Q022 모델 문서안내, 정답 절차안내, 보통, 예방정비가 끝나면 시운전 기록은 어디에 남기나요
  Q026 모델 긴급, 정답 문서안내, 엣지, 8호기가 조금 이상해서 우선 매뉴얼부터 보고 싶습니다
  Q033 모델 긴급, 정답 절차안내, 엣지, E-201 알람이 났을 때 임시 조치 방법을 알려 주세요


이번 실행에서는 98건이 스키마 검증과 업무 규칙 검증을 모두 통과했습니다. 통과율은 100%입니다. 그런데 정답 라벨과 비교하면 3건이 틀렸습니다. 세 건 모두 형식에는 흠이 없고, `category`는 허용값 안에 있으며, 업무 규칙에도 맞습니다. 값만 틀렸습니다. 2026-11-19에 두 번 실행했을 때는 3건과 4건이 틀렸습니다.

정확도는 정답 라벨과 결과가 일치한 건의 비율입니다. 이번 실행의 정확도는 96.9%(98건 중 95건)입니다. 스키마 검증 통과율과 정확도는 서로 다른 숫자이고, 따로 적어야 합니다. 결과 테이블을 받는 사람이 궁금한 것은 정확도인데, 정답 라벨 없이 계산할 수 있는 것은 통과율뿐이기 때문에 둘을 혼동하기 쉽습니다.

## 다시 실행하면 결과가 달라지는 문의

> **요점**
>
> 같은 문의를 한 번 더 분류해 결과가 달라진 건을 찾습니다. 달라진 건은 판단이 어려운 문의일 가능성이 높아 사람 확인 대상으로 삼습니다. 두 번 모두 같았다고 해서 정답인 것은 아닙니다.

재현성은 같은 입력으로 다시 실행했을 때 같은 결과가 나오는 정도입니다. 모델은 토큰을 확률에 따라 고르므로, 두 분류의 확률이 비슷한 문의는 실행할 때마다 결과가 달라질 수 있습니다. 이 성질을 이용하면 정답 라벨 없이도 "모델이 확신하지 못하는 건"을 찾을 수 있습니다.

**예제 2-8. 다시 실행해 분류가 달라진 건 찾기**

In [20]:
# 예제 2-8
second = run_batch(to_send)
first_by_id = {record['id']: record for record in first}
second_by_id = {record['id']: record for record in second}      # 순서가 아니라 id로 짝짓습니다
changed_ids = [row_id for row_id in first_by_id
               if first_by_id[row_id]['category'] != second_by_id[row_id]['category']]
print('두 번 실행해 분류가 달라진 건', len(changed_ids), '/', len(first))
for row_id in changed_ids:
    print(f"  {row_id} 1차 {first_by_id[row_id]['category']}, 2차 {second_by_id[row_id]['category']}"
          f", 정답 {answers.loc[row_id, 'category']}, {text_of[row_id][:30]}")

same = [record for record in first if record['id'] not in changed_ids and record['category']]
same_wrong = [record['id'] for record in same if not is_correct(record)]
print(f'두 번 모두 같았던 {len(same)}건 가운데 정답과 다른 건 {len(same_wrong)}건 {same_wrong}')

두 번 실행해 분류가 달라진 건 1 / 98
  Q043 1차 절차안내, 2차 문서안내, 정답 절차안내, 절차서 위치도 알고 싶고 승인 순서도 궁금합니다
두 번 모두 같았던 97건 가운데 정답과 다른 건 3건 ['Q022', 'Q026', 'Q033']


검증에 걸렸거나 분류가 달라진 건만 기본 모델로 다시 분류하고, 정답과 비교합니다.

In [21]:
# 예제 2-8 (걸린 건만 상위 모델로 다시 분류합니다)
flagged_ids = sorted(set(changed_ids) | {record['id'] for record in first if validate(record)})
rows_by_id = {row['id']: row for row in to_send}
rechecked = {row_id: classify(rows_by_id[row_id], model='기본') for row_id in flagged_ids}
for row_id, record in rechecked.items():
    print(f"  {row_id} 소형 1차 {first_by_id[row_id]['category']}, 소형 2차 {second_by_id[row_id]['category']}"
          f", 기본 {record['category']}, 정답 {answers.loc[row_id, 'category']}")
if flagged_ids:
    small_correct = sum(is_correct(first_by_id[row_id]) for row_id in flagged_ids)
    default_correct = sum(is_correct(record) for record in rechecked.values())
    print(f'걸린 {len(flagged_ids)}건의 정답 수: 소형 1차 {small_correct}, 기본 {default_correct}')
    print(f"다시 분류한 요금 ${total_cost(list(rechecked.values()), '기본'):.4f}")
else:
    print('다시 분류할 건이 없습니다.')

  Q043 소형 1차 절차안내, 소형 2차 문서안내, 기본 절차안내, 정답 절차안내
걸린 1건의 정답 수: 소형 1차 1, 기본 1
다시 분류한 요금 $0.0018


이번 실행에서는 98건 모두 두 번의 분류가 같았습니다. 그래서 두 번째 코드는 다시 분류할 건이 없다고 출력했습니다.

더 눈여겨볼 것은 첫 번째 코드 출력의 마지막 줄입니다. 두 번 모두 같았던 98건 가운데 3건(Q022, Q026, Q033)은 두 번 모두 같은 값으로 틀렸습니다. 재실행 비교는 1차에서 틀린 3건을 하나도 찾지 못했습니다. 다시 실행해도 결과가 같다는 것과 그 결과가 옳다는 것은 별개입니다.

분류가 달라지는 건이 몇 건인지는 실행마다 다릅니다. 2026-11-19에 실행했을 때는 98건 가운데 2건이 달라졌고, 두 건의 사정은 서로 반대였습니다.

- Q033은 1차에서 틀렸고 2차에서 맞았습니다. 재실행 비교가 틀린 건을 찾아낸 경우입니다.
- Q043은 1차에서 맞았는데 2차에서 다른 값이 나왔습니다. 문서의 위치와 승인 순서를 함께 물은, 두 라벨에 걸친 문의입니다. 틀린 건은 아니지만 모델이 확신하지 못하는 건이므로 사람이 확인할 만합니다.

그 실행에서는 걸린 2건을 기본 모델로 다시 분류했고, 2건 모두 정답과 같았습니다. 건수가 적어 이것만으로 기본 모델이 낫다고 말할 수는 없습니다. 요금은 $0.0057로, 98건을 소형 모델로 분류한 요금($0.0114)의 절반이었습니다. 걸린 건이 많아지면 상위 모델 재분류의 요금이 전체 요금보다 커집니다.

Q022와 Q026은 2026-11-19의 두 번의 실행과 이번 실행에서 모두 같은 값으로 틀렸습니다. 여러 번 실행해도 같은 값으로 틀린다면 우연이 아니므로, 모델보다 분류 기준을 먼저 의심합니다. Q022("시운전 기록은 어디에 남기나요")의 정답 라벨은 절차안내이지만, "어디에"라는 말 때문에 문서의 위치를 묻는 문서안내로도 읽힙니다. 지금의 규칙에는 이 둘을 구분하는 문장이 없습니다.

분류가 달라지는 건을 만났을 때 할 수 있는 일은 세 가지이고, 아래 순서로 검토합니다.

1. **분류 기준을 고칩니다.** 달라진 건을 직접 읽어 봅니다. 두 라벨에 걸친 문의라면 모델보다 기준이 모호한 것입니다. 이 예제의 규칙에는 "두 가지에 걸치면 긴급, 데이터요청, 절차안내, 문서안내 순으로 앞선 것을 고른다"는 우선순위가 있습니다. 이런 문장이 없거나, 있어도 해당 사례에 적용되지 않으면 문장을 더합니다. 달라지는 건이 많다면 라벨 체계 자체를 다시 설계합니다.
2. **상위 모델로 다시 분류합니다.** 걸린 건에만 쓰면 요금이 적게 듭니다. 다만 상위 모델의 응답이 정답이라는 보장은 없습니다. 정답 라벨이 있는 표본으로 효과를 확인한 뒤에 씁니다.
3. **사람에게 넘깁니다.** 기준을 고쳐도 달라지는 건, 틀리면 손실이 큰 건은 사람이 정합니다.

## 상태를 붙여 사람이 볼 건을 고르기

> **요점**
>
> 결과마다 auto, review, manual 가운데 하나를 붙입니다. 사람은 review와 manual만 봅니다. 그리고 auto로 넘긴 건 가운데 틀린 건이 얼마나 되는지를 정답 라벨로 확인합니다.

검증과 재실행 비교의 결과를 한 열로 모읍니다.

| 상태 | 뜻 | 해당하는 건 |
| --- | --- | --- |
| `auto` | 자동 처리. 사람이 보지 않고 다음 단계로 넘깁니다 | 검증을 통과했고 다시 실행해도 같았던 건 |
| `review` | 사람 확인. 모델의 결과는 있지만 사람이 확인합니다 | 업무 규칙에 걸린 건, 다시 실행해 분류가 달라진 건 |
| `manual` | 실패. 모델의 결과가 없어 사람이 처리합니다 | 보내지 않은 건, 호출 실패, 잘림, 거절, 스키마 불일치 |



*주황색 화살표는 검사에 걸린 건이 가는 길이고, 청록색 화살표는 세 검사를 모두 통과한 건이 가는 길입니다. 정답 라벨 비교는 이 그림에 없습니다. auto로 간 건이 실제로 맞았는지는 정답 라벨이 있는 표본으로 따로 확인합니다.*

**예제 2-9. 상태를 붙이고, 자동 처리한 건 가운데 틀린 건 세기**

In [22]:
# 예제 2-9
def status_of(record, changed_ids):
    """(상태, 사유)를 돌려줍니다."""
    why = validate(record)
    if record['error']:
        return 'manual', why                       # 모델의 결과가 없습니다
    if why:
        return 'review', why
    if record['id'] in changed_ids:
        return 'review', '재실행 결과 다름'
    return 'auto', None


result = pd.DataFrame(first)[['id', 'category', 'urgent']]
result['status'], result['why'] = zip(*[status_of(record, changed_ids) for record in first])
table = inquiries.merge(result, on='id', how='left')            # 원본 100행을 모두 남깁니다
not_sent = table['status'].isna()
table.loc[not_sent, 'status'] = 'manual'
table.loc[not_sent, 'why'] = '보내지 않음(빈 문의)'
print(table['status'].value_counts().to_dict())
print(table.loc[table['status'] != 'auto', ['id', 'category', 'status', 'why']].to_string(index=False))

{'auto': 97, 'manual': 2, 'review': 1}
  id category status          why
Q021      NaN manual 보내지 않음(빈 문의)
Q043     절차안내 review    재실행 결과 다름
Q063      NaN manual 보내지 않음(빈 문의)


`auto`로 넘긴 건이 실제로 맞았는지를 정답 라벨로 확인합니다.

In [23]:
# 예제 2-9 (자동 처리한 건의 정확도)
auto = table[table['status'] == 'auto']
auto_wrong = [row_id for row_id, category in zip(auto['id'], auto['category'])
              if category != answers.loc[row_id, 'category']]
print(f'auto {len(auto)}건, 정답과 다른 건 {len(auto_wrong)}건 {auto_wrong}')
print(f'auto의 정확도 {(len(auto) - len(auto_wrong)) / len(auto):.1%}')
all_wrong = [record['id'] for record in first if record['category'] and not is_correct(record)]
caught = sorted(set(all_wrong) - set(auto_wrong))
print(f'1차 실행에서 틀린 {len(all_wrong)}건 가운데 review로 잡힌 건 {len(caught)}건 {caught}')

auto 97건, 정답과 다른 건 3건 ['Q022', 'Q026', 'Q033']
auto의 정확도 96.9%
1차 실행에서 틀린 3건 가운데 review로 잡힌 건 0건 []


이번 실행에서 100건은 `auto` 98건, `manual` 2건(보내지 않은 빈 문의)으로 나뉘었습니다. 재실행에서 달라진 건이 없어 `review`는 0건입니다. 사람이 볼 건은 2건뿐이지만, `auto` 98건 가운데 3건이 정답과 달랐습니다. `auto`의 정확도는 96.9%입니다. 2026-11-19 실행에서는 `review` 2건이 생겼고, `auto` 96건의 정확도는 97.9%였습니다.

결과 테이블을 넘길 때는 이 두 숫자를 함께 넘깁니다. 이번 실행이라면 "사람이 2건만 확인하면 된다"와 "자동 처리한 98건에는 틀린 건이 약 3% 섞여 있다"입니다. 3%를 받아들일 수 있는 업무라면 이대로 쓰고, 받아들일 수 없다면 분류 기준을 고치거나 사람이 확인하는 범위를 넓힙니다.

문의 목록과 결과를 `id`로 결합할 때 원본의 행을 모두 남깁니다(`how='left'`). 보내지 않은 건과 실패한 건이 결과 테이블에서 사라지지 않고 `manual`로 남아야, 받는 사람이 전체 건수를 대조할 수 있습니다.

사람이 확인하는 방식은 세 가지가 있고, 틀렸을 때의 손실과 건수에 따라 고릅니다.

| 방식 | 사람이 보는 범위 | 이 방식을 쓰는 경우 |
| --- | --- | --- |
| 전수 확인 | 모든 건 | 안전이나 품질 판정처럼 틀리면 되돌리기 어려운 일 |
| 표본 확인 | 무작위로 고른 일부 | 정확도를 숫자로 말해야 할 때 |
| 걸린 건만 확인 | `review`와 `manual` | 건수가 많고 대부분이 단순할 때 |

예제 2-9의 첫 번째 코드가 걸린 건만 확인하는 방식입니다. 두 번째 코드는 표본 확인에 해당합니다. 다만 예제에는 정답 라벨이 이미 있어 `auto` 전체를 비교했고, 실제 업무에서는 `auto`에서 표본을 뽑아 사람이 정답을 붙입니다.

걸린 건만 확인하는 방식은 표본 확인과 함께 써야 합니다. 걸린 건만 보면 `auto`에 섞인 오류는 끝내 드러나지 않기 때문입니다. `auto`에서 표본을 뽑아 사람이 정답을 붙이고, 틀린 비율을 확인합니다. 그 비율이 업무에서 받아들일 수 있는 수준인지가 결과 테이블을 넘겨도 되는지의 기준입니다.

## 저장과 멱등성

> **요점**
>
> 결과에는 원본과 연결하는 id, 모델, 실행 시각, 상태를 함께 저장합니다. DB에는 id를 기본 키로 두어 같은 배치를 다시 넣어도 행이 늘지 않게 합니다.

### 결과 테이블에 남기는 열

결과 테이블을 받은 사람은 나중에 "이 값은 어떻게 나온 것인가"를 묻습니다. 아래 열이 있으면 대부분 답할 수 있습니다.

| 열 | 답할 수 있는 질문 |
| --- | --- |
| `id` | 원본의 어느 건인가. 문의 목록의 값을 그대로 씁니다 |
| 결과 필드(`category`, `urgent`) | 모델이 무엇이라고 답했는가 |
| `status`, `why` | 믿어도 되는가. 왜 사람 확인으로 넘어왔는가 |
| `model` | 어느 모델이 답했는가. 모델이 바뀌면 같은 문의도 결과가 바뀔 수 있습니다 |
| `run_at` | 언제 실행했는가. 같은 문의를 다시 분류한 결과와 구분합니다 |

**예제 2-10. 엑셀 두 시트와 SQLite에 저장하기**

In [24]:
# 예제 2-10
RUN_AT = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
table['model'] = llm.model_name('소형')
table['run_at'] = RUN_AT

os.makedirs('out', exist_ok=True)                       # 노트북 폴더에 결과 파일이 섞이지 않게 out 폴더에 둡니다
with pd.ExcelWriter('out/문의_분류결과.xlsx') as writer:
    table.to_excel(writer, sheet_name='전체', index=False)
    table[table['status'] != 'auto'].to_excel(writer, sheet_name='확인필요', index=False)

sheets = pd.read_excel('out/문의_분류결과.xlsx', sheet_name=None)
print({name: len(sheet) for name, sheet in sheets.items()})

{'전체': 100, '확인필요': 3}


같은 결과를 SQLite에 넣습니다. SQLite는 파이썬에 포함된 `sqlite3` 모듈로 쓰는 DB이고, 파일 하나가 DB 하나입니다. `id`를 기본 키로 두고, 같은 `id`가 이미 있으면 결과 열만 고칩니다.

In [25]:
# 예제 2-10 (SQLite에 넣고, 한 번 더 넣어 봅니다)
import sqlite3

if os.path.exists('out/inquiries.db'):
    os.remove('out/inquiries.db')                       # 예제를 다시 실행해도 같은 결과가 나오게 지웁니다
db = sqlite3.connect('out/inquiries.db')
db.execute('''CREATE TABLE classified (
    id TEXT PRIMARY KEY, category TEXT, urgent INTEGER, status TEXT,
    model TEXT, run_at TEXT, reviewed TEXT)''')

UPSERT = '''INSERT INTO classified (id, category, urgent, status, model, run_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                category = excluded.category, urgent = excluded.urgent, status = excluded.status,
                model = excluded.model, run_at = excluded.run_at'''
saved = table[table['category'].notna()]                # 보내지 않았거나 실패해 category가 없는 행은 넣지 않습니다
db_rows = [(row['id'], row['category'], int(row['urgent']), row['status'], row['model'], row['run_at'])
           for _, row in saved.iterrows()]
db.executemany(UPSERT, db_rows)
db.commit()
print('한 번 넣은 뒤 행 수', db.execute('SELECT COUNT(*) FROM classified').fetchone()[0])
db.executemany(UPSERT, db_rows)                         # 같은 결과를 한 번 더 넣습니다
db.commit()
print('두 번 넣은 뒤 행 수', db.execute('SELECT COUNT(*) FROM classified').fetchone()[0])
print(db.execute('SELECT status, COUNT(*) FROM classified GROUP BY status').fetchall())

한 번 넣은 뒤 행 수 98
두 번 넣은 뒤 행 수 98
[('auto', 97), ('review', 1)]


사람이 확인한 내용을 `reviewed` 열에 적어 둔 뒤 배치를 다시 넣으면 어떻게 되는지 두 방식을 비교합니다.

In [26]:
# 예제 2-10 (사람이 적은 값은 다시 넣어도 남는가)
review_id = db_rows[0][0]
db.execute("UPDATE classified SET reviewed = '확인함(담당자 A)' WHERE id = ?", (review_id,))
db.executemany(UPSERT, db_rows)
print('ON CONFLICT DO UPDATE 뒤:', db.execute(
    'SELECT reviewed FROM classified WHERE id = ?', (review_id,)).fetchone()[0])

REPLACE = '''INSERT OR REPLACE INTO classified (id, category, urgent, status, model, run_at)
             VALUES (?, ?, ?, ?, ?, ?)'''
db.executemany(REPLACE, db_rows)
print('INSERT OR REPLACE 뒤:', db.execute(
    'SELECT reviewed FROM classified WHERE id = ?', (review_id,)).fetchone()[0])
db.commit()

ON CONFLICT DO UPDATE 뒤: 확인함(담당자 A)
INSERT OR REPLACE 뒤: None


배치가 중간에 끊겼다면 이미 저장한 `id`를 건너뛰고 나머지만 보냅니다.

In [27]:
# 예제 2-10 (이미 처리한 id를 건너뜁니다)
done_ids = {row_id for (row_id,) in db.execute('SELECT id FROM classified')}
remaining = [row for row in to_send if row['id'] not in done_ids]
print(f'이미 저장한 건 {len(done_ids)}, 다음 실행에서 보낼 건 {len(remaining)}',
      [row['id'] for row in remaining])
db.close()

이미 저장한 건 98, 다음 실행에서 보낼 건 0 []


엑셀 파일에는 시트를 두 개 둡니다. "전체" 시트에는 원본의 모든 행이 있고, "확인필요" 시트에는 `auto`가 아닌 행만 있습니다. 사람은 "확인필요" 시트만 열면 됩니다.

> **주의**
>
> 결과 테이블에는 문의 원문이 함께 들어 있습니다. 원문에 개인정보나 공유하면 안 되는 내용이 있다면, 공유하는 파일에는 원문 열을 빼고 `id`와 결과 열만 넣습니다. 받은 사람은 `id`로 원본을 찾을 수 있습니다.

### 다시 실행해도 행이 늘지 않게

배치는 다시 실행하는 일이 잦습니다. 중간에 실패해서, 프롬프트를 고쳐서, 또는 실수로 두 번 실행해서입니다. 멱등성은 같은 작업을 여러 번 실행해도 결과가 한 번 실행한 것과 같은 성질입니다. 저장이 멱등하지 않으면 다시 실행할 때마다 같은 문의가 한 행씩 더 쌓이고, 분류별 건수를 셀 때 건수가 실제보다 크게 나옵니다.



*문의 100건 가운데 빈 문의 2건을 뺀 98건을 두 번 저장한 경우입니다. 주황색 상자의 방식은 행이 중복됩니다. 청록색 상자의 두 방식은 중복을 막고, 함께 쓸 수 있습니다.*

멱등하게 만드는 방법은 두 가지입니다.

- **결과를 새 값으로 바꾸려면 upsert를 씁니다.** 기본 키가 같은 행이 없으면 넣고, 있으면 고치는 저장 방식입니다. 프롬프트나 모델을 고친 뒤에 다시 실행할 때 씁니다.
- **끊긴 배치를 이어서 실행하려면 이미 저장한 `id`를 건너뜁니다.** 저장된 `id`는 보내지 않으므로 API 요금이 다시 들지 않습니다.

예제는 SQLite를 썼지만 다른 DB를 쓰더라도 원리는 같습니다. `id`를 기본 키로 두면 같은 `id`의 행을 두 개 넣을 수 없습니다.

SQLite에서 upsert를 하는 SQL 문은 두 가지이고, 같은 `id`의 행이 이미 있을 때의 동작이 다릅니다. 세 번째 코드의 결과가 그 차이를 보여 줍니다.

| SQL 문 | 같은 `id`가 이미 있을 때 | 사람이 `reviewed` 열에 적은 값 |
| --- | --- | --- |
| `INSERT … ON CONFLICT(id) DO UPDATE SET …` | `SET`에 적은 열만 고칩니다 | 남습니다 |
| `INSERT OR REPLACE …` | 기존 행을 지우고 새 행을 넣습니다 | 사라집니다(`None`) |

결과 테이블에 사람이 값을 적는 열이 있다면 `INSERT OR REPLACE`를 쓰지 않습니다. `INSERT OR REPLACE`는 기존 행을 지우고 새 행을 넣으므로, 배치를 다시 실행하는 순간 사람이 확인한 기록을 지웁니다.

엑셀 파일은 실행할 때마다 통째로 새로 씁니다. 같은 이름으로 쓰면 앞의 파일을 덮어쓰므로 행이 쌓이지는 않습니다. 여러 번의 실행 결과를 모아 두어야 한다면 엑셀이 아니라 DB에 저장합니다.

## 실행 조건 기록

> **요점**
>
> 결과 파일 옆에 그 결과를 만든 조건을 파일로 남깁니다. 모델, 프롬프트, 스키마, 건수가 들어갑니다.

같은 문의 목록이라도 모델, 프롬프트, 스키마 가운데 하나가 바뀌면 결과가 바뀝니다. 결과 파일만 남아 있으면 몇 주 뒤에 그 숫자가 어떤 조건에서 나왔는지 알 수 없습니다. 실행할 때마다 조건을 파일 하나로 남깁니다.

**예제 2-11. 실행 조건을 파일로 남기기**

In [28]:
# 예제 2-11
import hashlib

meta = {
    'run_at': RUN_AT,
    'model': llm.model_name('소형'),
    'structured_mode': llm.structured_mode(),
    'rules_sha1': hashlib.sha1(RULES.encode('utf-8')).hexdigest()[:12],   # 프롬프트가 바뀌었는지 확인하는 값
    'rules_head': RULES[:30] + '…',
    'schema_fields': list(Inquiry.model_fields),
    'n_input': len(inquiries),
    'n_sent': len(to_send),
    'status': table['status'].value_counts().to_dict(),
    'input_tokens': sum(record['input_tokens'] for record in first),
    'output_tokens': sum(record['output_tokens'] for record in first),
}
with open('out/run_meta.json', 'w', encoding='utf-8') as file:
    json.dump(meta, file, ensure_ascii=False, indent=1)
for key, value in meta.items():
    print(f'{key:16s}', value)

run_at           2026-11-26 09:18
model            claude-haiku-5-5
structured_mode  json_schema
rules_sha1       0b2b6f443b45
rules_head       너는 반도체 공장 현장에서 들어오는 문의를 분류하는 도…
schema_fields    ['id', 'category', 'urgent']
n_input          100
n_sent           98
status           {'auto': 97, 'manual': 2, 'review': 1}
input_tokens     77244
output_tokens    5922


프롬프트는 전체를 적어 두어도 되고, 위처럼 해시값만 남겨도 됩니다. 해시값은 글이 한 글자라도 바뀌면 달라지므로, 두 실행이 같은 프롬프트를 썼는지 비교할 수 있습니다. 프롬프트 파일을 버전 관리 도구로 관리하고 있다면 그 버전을 적습니다.

## 실습

실습은 `2일차_02_실습.ipynb`에서 합니다. 재료는 접수 요청 80건(`접수_실습.csv`)과 담당팀 기준(`접수_규칙.txt`)입니다. 접수 본문을 담당팀(설비, 시설, IT, 안전, 기타), 위치, 긴급 여부로 정리합니다. 80건 모두에 정답 라벨(`접수_실습_정답.csv`)이 있습니다. 실습마다 테스트 셀이 있습니다. 테스트 셀은 mock 호출을 쓰므로 요금이 들지 않습니다. 실행 셀은 실제 API를 부릅니다(실습 5는 3건, 실습 6은 80건, 실습 7은 80건씩 세 번). 풀이와 해설은 `2일차_02_실습_풀이.ipynb`에 있습니다.

**실습 5. 스키마 설계**

접수 본문을 읽고 담당팀, 위치, 긴급 여부를 받는 pydantic 클래스 `Ticket`을 작성합니다. 위치는 본문에 없을 수 있습니다. 3건을 먼저 보내 결과를 확인합니다.

완료 기준: 테스트 셀이 "실습 5 통과"를 출력합니다. 3건의 결과에서 위치가 없는 접수가 `None`으로 오는지 확인합니다.

**실습 6. 배치 처리와 멱등한 저장**

한 건을 처리하는 `classify()`와, 결과를 SQLite에 저장하는 `save()`, 저장하지 않은 건만 골라 보내는 `run()`을 작성합니다. 일부가 실패한 뒤 다시 실행하면 실패한 건만 보내야 합니다.

완료 기준: 테스트 셀이 "실습 6 통과"를 출력합니다. 80건을 실제로 처리해 저장한 행 수를 확인하고, 한 번 더 실행했을 때 보내는 건이 0건인지 확인합니다.

**실습 7. 검증, 상태, 정확도**

업무 규칙을 넣은 `validate()`를 작성하고 상태를 붙입니다. 모델이 적은 위치가 본문에 실제로 나오는 문자열인지도 검사합니다. 정답 라벨과 비교하는 측정 셀을 실행해 담당팀의 정확도와, `auto`로 분류된 건 가운데 틀린 건수를 확인합니다. 그다음 두 가지를 바꿔 다시 실행하고 숫자를 비교합니다. 담당팀 허용값에서 "IT"를 뺀 경우와, 규칙에서 긴급 기준 문장을 뺀 경우입니다.

완료 기준: 테스트 셀이 "실습 7 통과"를 출력합니다. 스키마 검증 통과율, 업무 규칙에 걸린 건수, 담당팀 정확도, `auto` 가운데 틀린 건수를 적습니다. 두 실험에서 바뀐 숫자와 그 이유를 한 문장씩 적습니다.

풀이를 실행한 결과는 다음과 같습니다. 여러분의 숫자와 비교해 봅니다.

| 실습 | 풀이 실행 결과 (2026-11-19, Claude 모델) |
| --- | --- |
| 실습 5 | 테스트 7개 통과. 3건 가운데 위치가 없는 접수 1건의 `location`이 `None`으로 옴 |
| 실습 6 | 테스트 8개 통과. 첫 실행에서 80건을 보내 80행 저장, 다시 실행했을 때 보낸 건 0건 |
| 실습 7 실험 전 | 2026-11-23에 다시 실행한 결과입니다(실험 A, B도 같습니다). 스키마 검증 통과 80/80, 업무 규칙에 걸린 건 0건, 담당팀 정확도 76/80, 긴급 여부 정확도 79/80. `auto` 80건 가운데 담당팀이 틀린 건 4건 |
| 실습 7 실험 A | "IT"를 빼면 정답이 IT인 16건이 기타 12건, 시설 4건으로 분류됨. 담당팀 정확도 64/80. 스키마 검증 통과는 그대로 80/80 |
| 실습 7 실험 B | 긴급 기준 문장을 빼면 `urgent`가 `true`인 건이 22건에서 27건으로 늘고 긴급 여부 정확도가 79/80에서 74/80으로 내려감 |

실험 A와 B에서 스키마 검증 통과율은 바뀌지 않았습니다. 허용값과 기준 문장이 결과를 바꿨는데도, 형식만 보는 검증으로는 그 차이를 알 수 없습니다.

## 핵심 정리

- 모델의 응답은 문자열입니다. 응답에서 값을 꺼내 조건문에 쓰거나 저장하려면 정해진 구조로 받아야 하고, 프롬프트로 형식을 요청하는 것만으로는 그 구조가 보장되지 않습니다.
- 구조화 출력은 스키마에 어긋나는 토큰을 생성 단계에서 제외합니다. 정상 종료한 응답은 스키마에 맞는 JSON입니다. 잘리거나 거절된 응답은 그렇지 않으므로 종료 이유를 계속 확인합니다.
- 구조화 출력이 지키는 것은 형식입니다. 값이 옳은지, 다시 실행해도 같은지, 필드 값이 서로 모순되지 않는지는 지켜 주지 않습니다.
- 결과 테이블에 필요한 열을 먼저 정하고, 그 열에 맞춰 스키마를 작성합니다. 골라야 하는 값은 enum으로 좁히고 나머지를 받는 값을 두며, 없을 수 있는 값은 null을 허용합니다.
- 검증은 네 단계입니다. 스키마 검증, 업무 규칙 검증, 재실행 비교로 사람이 볼 건을 고르고, 정답 라벨 비교로 자동 처리한 건의 정확도를 확인합니다.
- 결과에는 id, 상태, 사유, 모델, 실행 시각을 함께 저장하고, 실행 조건을 파일로 남깁니다.
- 저장은 멱등하게 만듭니다. 기본 키와 upsert로 중복을 막고, 저장한 id를 건너뛰어 끊긴 배치를 이어 갑니다.

## 자가점검

**1.** `category`를 enum으로 정한 스키마로 구조화 출력을 받았습니다. 응답이 정상 종료했다면 이것만으로 보장되는 것은 무엇입니까?

- A. 분류가 사람이 정한 정답과 같다
- B. `category`가 목록의 값 중 하나이고 응답이 JSON으로 파싱된다
- C. 같은 문의를 다시 보내도 같은 분류가 나온다

<details><summary>정답 보기</summary>

**B.** 구조화 출력은 형식을 보장합니다. 목록 안에서 어느 값을 고를지는 모델이 확률에 따라 정하므로 A와 C는 보장되지 않습니다. "구조화 출력" 절을 다시 봅니다.

</details>

**2.** 프롬프트에 "JSON 한 줄만 쓴다"고 적는 것과 구조화 출력은 형식을 지키는 방법이 어떻게 다른지 설명해 보세요.

<details><summary>정답 보기</summary>

프롬프트로 요청하면 형식에 맞는 토큰의 확률이 높아질 뿐이고, 다른 토큰이 뽑힐 가능성이 남습니다. 구조화 출력은 스키마에 어긋나는 토큰을 후보에서 제외한 뒤 남은 토큰 중에서 고르므로, 형식이 규칙으로 지켜집니다.

</details>

**3.** 구조화 출력으로 받은 응답을 `json.loads`로 읽다가 오류가 났습니다. 가장 먼저 확인할 것은 무엇입니까?

<details><summary>정답 보기</summary>

종료 이유입니다. 최대 출력 토큰에 도달해 잘렸거나(`max_tokens`, OpenAI 호환은 `length`) 거절된 응답은 구조화 출력에서도 스키마에 맞지 않을 수 있습니다. 잘렸다면 최대 출력 토큰을 올려 재시도합니다(예제 2-3).

</details>

**4.** 스키마의 `location`을 `{'type': ['string', 'null']}`로 두었습니다. `'null'`을 빼고 `'string'`만 두면, 본문에 위치가 없는 접수는 어떻게 됩니까?

- A. 그 건은 오류가 나서 결과를 받지 못한다
- B. 모델이 빈 문자열이나 "없음" 같은 말, 또는 본문에 없는 위치를 채울 수 있다
- C. `location` 필드가 응답에서 자동으로 빠진다

<details><summary>정답 보기</summary>

**B.** 필수 필드의 자료형이 문자열뿐이면 모델은 어떤 문자열이든 채워야 합니다. 오류는 나지 않습니다. 그래서 값이 없다는 것을 코드에서 한 가지 조건으로 확인할 수 없고, 지어낸 값이 들어갈 수 있습니다(예제 2-4).

</details>

**5.** 담당팀 허용값이 설비, 시설, 안전 세 가지뿐인 스키마로 "노트북 배터리 교체" 요청을 분류하면 어떤 일이 생깁니까? 이를 막으려면 허용값을 어떻게 두어야 합니까?

<details><summary>정답 보기</summary>

오류 없이 세 값 중 하나로 분류됩니다. 모델은 목록 안에서 반드시 하나를 골라야 하기 때문입니다. 목록에 "기타"처럼 나머지를 받는 값을 두고, 그 값으로 분류된 건은 사람이 확인합니다.

</details>

**6.** 구조화 출력으로 받은 100건이 스키마 검증을 모두 통과했습니다. 이 결과 테이블을 그대로 다른 사람에게 넘기지 않는 이유는 무엇입니까?

- A. 구조화 출력은 JSON을 보장하지 않아서
- B. 엑셀 시트가 둘이라서
- C. 형식이 맞아도 값이 틀렸거나, 업무 규칙에 어긋나거나, 다시 실행하면 바뀌는 건이 있어서

<details><summary>정답 보기</summary>

**C.** 스키마 검증 통과율은 형식이 맞는 건의 비율이고, 정확도와는 다른 숫자입니다. 업무 규칙 검증과 재실행 비교로 사람이 볼 건을 고르고, 정답 라벨이 있는 표본으로 정확도를 확인한 뒤에 넘깁니다.

</details>

**7.** 같은 문의 100건을 두 번 분류했더니 97건의 분류가 같았습니다. "97건은 맞았다"고 말할 수 없는 이유를 설명해 보세요.

<details><summary>정답 보기</summary>

재실행 비교는 실행마다 결과가 바뀌는지를 볼 뿐, 옳은지를 보지 않습니다. 분류 기준이 모호하거나 모델이 일관되게 잘못 이해한 문의는 두 번 모두 같은 값으로 틀립니다. 옳고 그름은 정답 라벨과 비교해야 알 수 있습니다(예제 2-8).

</details>

**8.** `id`를 기본 키로 둔 DB 테이블에 같은 80건을 upsert로 두 번 넣었습니다. 테이블의 행 수는 몇 개입니까? 기본 키 없이 끝에 추가하는 방식이었다면 몇 개입니까?

<details><summary>정답 보기</summary>

upsert는 80행, 끝에 추가하는 방식은 160행입니다. 160행이 된 테이블에서 분류별 건수를 세면 모든 숫자가 두 배로 나옵니다.

</details>

**9.** 결과 테이블에 담당자가 확인 내용을 적는 열이 있습니다. 배치를 다시 실행해 저장할 때 `INSERT OR REPLACE`를 쓰면 안 되는 이유는 무엇입니까?

<details><summary>정답 보기</summary>

`INSERT OR REPLACE`는 같은 기본 키의 행을 지우고 새 행을 넣습니다. 새 행에 없는 열은 비게 되므로 담당자가 적은 값이 사라집니다. 고칠 열만 지정하는 `ON CONFLICT … DO UPDATE SET`을 씁니다(예제 2-10).

</details>

**10.** 사람은 `review`와 `manual`로 분류된 건만 확인하기로 했습니다. 이 방식만으로는 알 수 없는 것이 무엇이고, 그것을 알려면 무엇을 더 해야 합니까?

<details><summary>정답 보기</summary>

`auto`로 넘어간 건 가운데 틀린 건이 얼마나 되는지 알 수 없습니다. `auto`에서 표본을 뽑아 사람이 정답을 붙이고 정확도를 계산합니다. 그 정확도가 업무에서 받아들일 수 있는 수준인지 확인한 뒤에 결과를 넘깁니다.

</details>